# 11 — Export results tables

Save machine-learning performance summaries and biological-feature ablation tables separately from plotting notebooks. This only reads existing outputs.

In [ ]:
from pathlib import Path
import json
import pandas as pd
def find_root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/"config"/"genes.json").exists():return p
    raise FileNotFoundError("Project root not found")
ROOT=find_root()
results=ROOT/"results"
reports=ROOT/"reports"
reports.mkdir(exist_ok=True)
def stack(pathpattern):
    paths=sorted(results.glob(pathpattern))
    pieces=[]
    for p in paths:
        t=pd.read_csv(p)
        if "experiment_name" not in t:
            t["experiment_name"]=p.stem.replace("_metrics","").replace("_history","")
        pieces.append(t)
    return pd.concat(pieces,ignore_index=True) if pieces else pd.DataFrame()
metrics=stack("metrics/exp*_metrics.csv")
history=stack("histories/exp*_history.csv")
abl_path=results/"metrics"/"feature_group_ablation_metrics.csv"
ablation=pd.read_csv(abl_path) if abl_path.exists() else pd.DataFrame()
if metrics.empty:
    print("No saved model metrics found; export skipped.")
else:
    summary=(metrics.groupby("experiment_name",as_index=False)
             .agg(mean_r2=("r2","mean"),mean_mae=("mae","mean"),mean_rmse=("rmse","mean")))
    best=metrics.loc[metrics.groupby("target")["r2"].idxmax()].sort_values("target")
    summary.to_csv(reports/"experiment_summary.csv",index=False)
    metrics.to_csv(reports/"per_target_metrics.csv",index=False)
    best.to_csv(reports/"best_model_per_target.csv",index=False)
    print(summary.to_string(index=False))
    try:
        with pd.ExcelWriter(reports/"tcga_taa_experiment_summary.xlsx",engine="openpyxl") as writer:
            summary.to_excel(writer,sheet_name="model_summary",index=False)
            metrics.to_excel(writer,sheet_name="per_target_metrics",index=False)
            best.to_excel(writer,sheet_name="best_per_target",index=False)
            if not history.empty: history.to_excel(writer,sheet_name="training_history",index=False)
            if not ablation.empty: ablation.to_excel(writer,sheet_name="feature_ablation",index=False)
        print("Excel workbook saved")
    except ImportError:
        print("Install openpyxl for optional Excel export; CSV files were saved")
